# Part 1: Recognition

In this part of the assignment, you will implement and train neural networks, including convolutional neural networks, for an image recognition task using PyTorch. Specifically, we will classify color images of animals, vehicles, and other objects by predicting a label for the object type.

**Learning objectives.** You will:
1. Define multilayer perceptrons and convolutional neural networks using PyTorch
2. Optimize neural networks using automatic differentation and minibatch stochastic gradient descent in PyTorch
3. Evaluate different learning hyperparameters and model architecture choices by evaluating validation performance
4. Accelerate neural network training and inference using a graphics processing unit (GPU) with software support in PyTorch

## Getting Started

We recommend that you start by reviewing the extremely concise [PyTorch Quickstart tutorial](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html), which will help familiarize you with the API and the tasks below. You can also review more specific concepts in more detail beyond the quickstart -- there are additional tutorial pages with more details on tensors, Datasets and Dataloaders, etc.

Once you are ready to get started experimenting yourself, run the following code to import relevant PyTorch modules, download the dataset, split into a train, validation, and test dataset, and prepare PyTorch dataloaders for batching.

In [ ]:
# Run but DO NOT MODIFY this code

# Import libraries
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np

# Set a random seed for reproducibility
torch.manual_seed(2025)

# Load CIFAR-10 dataset
train_data = datasets.CIFAR10(root="data", train=True, download=True, transform=ToTensor())
train_set, val_set = random_split(train_data, [0.8, 0.2])
test_data = datasets.CIFAR10(root="data", train=False, download=True, transform=ToTensor())

# Prepare DataLoaders
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

Then run the following code to visualize some random examples of the training data. You will see that you are working with color images of objects belonging to one of ten labeled classes. Our goal will be to build predictive models that take an image as input and predict the class to which they belong (e.g., is this an image of a cat or a truck?)

In [ ]:
# Run but DO NOT MODIFY this code

# Visualize Random Examples
classes = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog",
           "horse", "ship", "truck"]

# Function to show an image
def imshow(img):
    img = img / 2 + 0.5  # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')

# Get some random training images
dataiter = iter(torch.utils.data.DataLoader(train_data, batch_size=10, shuffle=True))
images, labels = next(dataiter)

# Create a grid of subplots
fig, axs = plt.subplots(2, 5, figsize=(10, 4))
fig.suptitle('Examples from CIFAR-10 train dataset', fontsize=12)

# Plot 10 images
for i, ax in enumerate(axs.flat):
    ax.imshow(np.transpose(images[i].numpy(), (1, 2, 0)))
    ax.set_title(f"{classes[labels[i]]}")
    ax.axis('off')

plt.tight_layout()
plt.show()

Run the following code to define a helper function for visualizing your model training and validation performance.

In [ ]:
# Run but DO NOT MODIFY this code

from matplotlib import pyplot as plt

# Helper function to visualize performance during training
def plot_training_curves(train_losses, val_accuracies):
    """Plot training loss and validation accuracy curves.
    
    Parameters
    ----------
    train_losses : list of float
        Training loss values for each epoch. Should have one value per epoch.
    val_accuracies : list of float
        Validation accuracy values for each epoch. Should have same length as
        train_losses. Accuracy values should be between 0 and 1 (or 0 and 100
        if using percentages).
        
    Returns
    -------
    None
        Displays matplotlib figure with two subplots showing training curves.
        
    Examples
    --------
    >>> train_losses = [0.8, 0.6, 0.4, 0.3, 0.2]
    >>> val_accuracies = [0.75, 0.80, 0.85, 0.87, 0.88]
    >>> plot_training_curves(train_losses, val_accuracies)
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(train_losses)
    ax1.set_title('Training Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.grid(True)
    
    ax2.plot(val_accuracies)
    ax2.set_title('Validation Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

## Task 1

In this task you will build and train an MLP classifier using PyTorch and CPU compute.

**1. Define a multilayer perceptron [using PyTorch](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)** to classify the CIFAR-10 images. Your architecture should:
  - [Flatten](https://docs.pytorch.org/docs/stable/generated/torch.flatten.html) the 32×32×3 input images to vectors of size 3072
  - Contain at least two [linear layers](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) (i.e., at least one hidden layer plus one output layer). The basic structure is: **input → hidden → output**. You may use a deeper model with additional hidden layers (like two hidden layers + one output layer), but this is not required. 
  - Use nonlinear activations ([ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html) recommended) for hidden layers
  - Have 10 output units for the classification task
  - Use no more than 10,000,000 total model parameters. You will likely want to use hidden layers with hundreds or even thousands of hidden units for this larger input size.

**2. Train your model** to minimize the [Cross entropy loss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) on the training data. For this task simply use the CPU for training (this is the default and does not require you to do anything). For training, you should:
  - Use the [SGD optimizer](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html) that implements standard minibatch stochastic gradient descent, or the [Adam optimizer](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html). You may need to adjust hyperparameters (e.g., for the learning rate, momentum, etc.) but can start with the default values.
  - Evaluate and record the training loss and validation accuracy once per epoch so that you can visualize training performance using the `plot_training_curves` function defined above.
  - Implement early stopping so that your model stops training if the validation accuracy does not improve for a few consecutive epochs (a concept known as 'patience'). While you can set the patience value to 1 (stop after the first epoch with no improvement), you might get better results with a patience of 2 or 3 to avoid stopping prematurely due to minor fluctuations. You may also want to set a maximum number of epochs (e.g., 10 or 20) to ensure that the code terminates in a reasonable amount of time even if training hyperparameters are suboptimal.
  - Record the total amount of time (in seconds) to train the model and the average time per epoch (you will be asked to use the average time per epoch again later in task 2 for a fair comparison). You can use Python's [`time()` functionality](https://docs.python.org/3/library/time.html#time.time).
  - Note that the `train_loader` and `val_loader` defined above can be used for automatic batching and shuffling the training and validation data during training. 
  - Note that the `test_loader` should not be used in this task.
    
**Your goal is to achieve a validation accuracy of 45% or better using no more than 10,000,000 total model parameters.** Note that there are ten classes so random guessing with balanced data would only achieve 10% accuracy in expectation.

Once you reach this goal, **report the following:**
  - Use the `plot_training_curves` function to visualize the training results
  - Report the total number of model parameters. Show your work for the number of model parameters, either by showing your calculations or showing the code you used to count the number of model parameters ([The .parameters() method will return an iterable over the model parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.parameters)).
  - Report the total amount of time that was required to train your model in seconds.
  - Report the average time per epoch in seconds.

In [ ]:
# Write code for task 1 here

# Setting a random seed for reproducibility
torch.manual_seed(2025)

import torch.nn as nn
import time

class MLP(torch.nn.Module):
    """Definition of a multilayer perceptron for classification 
    in Pytorch, inheriting from the torch.nn.Module base class."""
    def __init__(self):
        super(MLP, self).__init__()
        # TODO: finish constructor/initialization


    def forward(self, x):
        """ Compute logits for batch x by forward propagation.
        Parameters
        ----------
        x : tensor, shape = [n_examples, n_channels, width, height]
        """
        # TODO: implement forward propagation

*For Task 1: report the number of model parameters, show work, total training time, and average time per epoch here*

| Metric | Value |
|--------|-------|
| **Total Parameters** | |
| **Work/Code Used** | |
| **Total Training Time (seconds)** | |
| **Average Time per Epoch (seconds)** | |

## Task 2

Train the **exact same** multilayer perceptron architecture that you defined in Task 1, exactly as you trained it in Task 1, but this time **use GPU compute to accelerate the training.** A GPU is a Graphics Processing Unit and is specialized to efficiently compute intensive but highly parallel operations such as matrix multiplications.

If you are running this code on the Google Colab or a CS Department Compute Cluster OnDemand, you should have access to a CUDA enabled GPU device. In both cases you will need to specifically request a GPU. Using OnDemand, you should request a device from gpu partition and request 1 GPU. To change the Runtime in Google Colab, on the top drop-down menu select Runtime, then select Change runtime type. Under Hardware accelerator, select T4 GPU, then click Save.

See the [CUDA semantics documentation](https://pytorch.org/docs/stable/notes/cuda.html#cuda-semantics) for details, or the quick tips for common operations below.
  - Check GPU availability: `torch.cuda.is_available()`
  - Create device object: for example: `torch.device('cuda' if torch.cuda.is_available() else 'cpu')`
  - Move model and data to device during training: for example `model.to(device)` and `data.to(device)`

If you are running on your own local device, you may or may not have a GPU available -- it is your responsibility to manage your own device or you can complete the assignment on one of the two above browser-based cloud solutions if you prefer. The CUDA backend referenced here is for NVIDIA GPUs. Most modern macbooks using Apple silicon processors have GPU availability with the [PyTorch MPS backend](https://docs.pytorch.org/docs/stable/notes/mps.html).

Your goal in this task is to **achieve similar results as in part 1 but with substantially less training time.**

Verify that you still achieve at least 45% accuracy and then **report the speedup factor** of your GPU training, measured using average time per epoch for a fair comparison:

$$ \frac{\text{Average time per epoch for CPU training in seconds}}{\text{Average time per epoch for GPU training in seconds}} $$

**Note:** Depending on your execution environment, hardware, and model size, you may experience different speedup factors or possibly no speedup at all, and **will not be penalized** for your empirical timing results as long as you have a correct implementation.

In [ ]:
# Run this cell to check GPU availability
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("No GPU available.")

In [ ]:
# Write code for task 2 here

# Setting a random seed for reproducibility
torch.manual_seed(2025)


*Task 2: Report speedup factor here*

| Metric | Value |
|--------|-------|
| **CPU Average Time per Epoch (seconds)** | |
| **GPU Average Time per Epoch (seconds)** | |
| **Speedup Factor** | |

## Task 3

In this task you will show that a Convolutional Neural Network can use fewer parameters to train a better model. CNNs often achieve better performance with fewer parameters for computer vision tasks due to parameter sharing and translation invariance.

**1. Define a convolutional neural network [using PyTorch](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)** to classify the CIFAR-10 images. Your architecture should:
  - Contain at least one [convolutional layer](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) followed by nonlinear activations such as the [ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html) (you can use a deeper model with additional hidden layers but are not required to do so). 
      - You should experiment with different numbers of filters / out channels and kernel sizes, but common values include 8-32 channels and 3x3, 5x5, or 7x7 kernels. 
  - Contain at least one [pooling layer](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) (you can use a deeper model with additional hidden layers but are not required to do so)
      - You should experiment with different kernel sizes, but common values include 2x2 or 3x3
  - [Flatten](https://docs.pytorch.org/docs/stable/generated/torch.flatten.html) the output from the convolutional and pooling layers to vectors
  - Contain at least two [linear layers](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) (i.e., at least one hidden layer plus one output layer). The basic structure is: **input → conv → hidden → output**. You can use a deeper model with additional hidden layers but are not required to do so.
  - Use nonlinear activations ([ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html) recommended) for hidden layers
  - Have 10 output units for the classification task
  - **Use fewer total model parameters than the MLP you used in Tasks 1-2.** The goal is to demonstrate that CNNs can achieve better performance with fewer parameters.
  - Note that the `test_loader` should not be used in this task.

**2. Train your model** as in Task 1, utilizing GPU compute as in Task 2. You could reuse your training function from Task 1.
    
**Your goal is to achieve a validation accuracy of 60% or better using fewer total model parameters than your MLP from Tasks 1-2.**

Once you reach this goal, **report the following:**
  - Use the `plot_training_curves` function to visualize the training results
  - Report the total number of model parameters. Show your work for the number of model parameters, either by showing your calculations or showing the code you used to count the number of model parameters ([The .parameters() method will return an iterable over the model parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.parameters)).
  - Report the total amount of time that was required to train your model in seconds.
  - Report the average time per epoch in seconds.

In [ ]:
# Write code for task 3 here

# Setting a random seed for reproducibility
torch.manual_seed(2025)

class ConvNN(torch.nn.Module):
    """Definition of a convolutional neural network for classification 
    in Pytorch, inheriting from the torch.nn.Module base class."""
    def __init__(self):
        super().__init__()
        # TODO: finish constructor/initialization


    def forward(self, x):
        """ Compute logits for batch x by forward propagation.
        Parameters
        ----------
        x : tensor, shape = [n_examples, n_channels, width, height]
        """
        # TODO: implement forward propagation

*For Task 3: report the number of model parameters, show work, and report training time here*

| Metric | Value |
|--------|-------|
| **CNN Total Parameters** | |
| **MLP Total Parameters (from Task 1)** | |
| **Work/Code Used** | |
| **Total Training Time (seconds)** | |
| **Average Time per Epoch (seconds)** | |

## Task 4

In this task you will use dropout regularization during training to improve the generalization of your convolutional neural network.

1. Before adding dropout, let's establish a baseline. Take your best model from Task 3 (the one that achieved the highest validation accuracy) and evaluate its performance on the `test_loader` (which has not been used until now), to obtain the test accuracy **without Dropout**. Report this test accuracy.

2. Second, modify your convolutional neural network architecture that you defined in Task 3 to **use Dropout to improve generalization.** Specifically, add a [PyTorch Dropout layer](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html) after the hidden linear layer that comes right before the output layer with a drop rate `p = 0.2` (this is lower than the default). Then train the model exactly as you did in Task 3.

3. Finally, evaluate the the model of Step 2 with `test_loader` and report the test accuracy. Then report the accuracy difference before and after applying Dropout.

**Note:** It is possible you will not see an improvement in generalization performance when incorporating dropout, especially if your convolutional network from task 3 was not overfitting (e.g., a small network may actually underfit on the data). You **will not be penalized** as long as you have a correct implementation.

In [ ]:
# Write code for task 4 here

# Setting a random seed for reproducibility
torch.manual_seed(2025)

*For Task 4: Report **test accuracies** with and without dropout, and provide the test accuracy difference here.*

| Metric | Value |
|--------|-------|
| **Test Accuracy WITHOUT Dropout** | |
| **Test Accuracy WITH Dropout** | |
| **Accuracy Difference** | |